## create_agent

- `create_agent` is the LangChain 1.x way of building the loop
- `response_format` makes it return a typed object instead of prose
- A checkpointer gives it memory across separate calls

Three parts : a tool-using agent, structured output with Pydantic, and
short-term memory.

### Installing the libraries

In [ ]:
!pip install -q langchain_mcp_adapters "mcp>=2"

Run the next cell first. In Colab, add the course key under the key
icon in the left sidebar as `COURSE_API_KEY`, with *Notebook access* on.

In [ ]:
# Course setup. In Colab: add the course key under the key icon (left sidebar)
# as COURSE_API_KEY. On your own machine it uses Ollama instead.
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Pinned, including the transitive ones. Unpinned, pip takes whatever
    # shipped this morning : a newer core moves ModelError, and a newer openai
    # rejects this endpoint's usage payload. Keep in step with requirements.txt.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "langchain==1.3.9", "langchain-core==1.4.7",
                    "langchain-openai==1.3.2", "langchain-classic==1.0.8",
                    "langgraph==1.2.5", "openai==2.41.1", "python-dotenv"],
                   check=True)
    MODEL = "qwen3.8-flash"          # try qwen3.8-max too
    BASE = "https://token-plan.ap-southeast-1.maas.aliyuncs.com/compatible-mode/v1"
    THINKING = {"enable_thinking": False}
    KEY = os.getenv("COURSE_API_KEY")
    if not KEY:
        try:
            from google.colab import userdata
            KEY = userdata.get("COURSE_API_KEY")   # raises if unset or not shared
        except Exception:
            import getpass
            KEY = getpass.getpass("Course key (paste the one from the trainer): ")
else:
    from dotenv import load_dotenv
    load_dotenv()
    MODEL = "qwen3.5:2b"             # try qwen3.5:4b too
    BASE = "http://localhost:11434/v1"
    KEY = "ollama"
    THINKING = {"reasoning_effort": "none"}

from langchain_openai import ChatOpenAI


def make_llm(**kw):
    kw.setdefault("temperature", 0)
    kw.setdefault("model", MODEL)
    kw.setdefault("extra_body", THINKING)
    return ChatOpenAI(base_url=BASE, api_key=KEY, **kw)


def make_embeddings(**kw):
    # The course endpoint has no embeddings, so they run here instead. 90 MB,
    # installed only by the notebooks that actually ask for them.
    try:
        from langchain_huggingface import HuggingFaceEmbeddings
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "langchain-huggingface==0.3.1"], check=True)
        from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


DATA_URL = ("https://raw.githubusercontent.com/FeikoWielsma/"
            "Building-AI-Agents/main/data/")


def data(name):
    """Path to a course data file. Downloads it in Colab, local copy otherwise."""
    if os.path.exists(f"../data/{name}"):
        return f"../data/{name}"
    if not os.path.exists(name):
        import urllib.request
        urllib.request.urlretrieve(DATA_URL + name, name)
    return name


llm = make_llm()
print(f"model={MODEL}")

### Exercise create_agent
Task: Modify the code below so that the tool used, instead of rating a city, returns a recipe for a chosen dish. We can use a dictionary and add recipes for several dishes. If the dish is not among the dictionary keys, return the recipe for "water for tea'.

In [ ]:
from langchain.agents import create_agent

RECIPES = {
    "lasagna": "Layer pasta sheets with ragu and bechamel, top with cheese, bake 40 minutes.",
    "pancakes": "Mix 200g flour, 2 eggs and 300ml milk. Fry two minutes a side.",
}


def recipe_for_dish(dish: str) -> str:
    """Returns the itinerary for a chosen trip."""
    return RECIPES.get(dish, "Boil water. Add a tea bag. Wait three minutes.")


def open_cmd(command: str) -> str:
    """Runs a shell command on this machine."""
    # This tool is deliberately not wired to a shell.
    #
    # The docstring above is the whole of what the model sees, and it is enough
    # for the model to call this with any string it likes. The original version
    # of this exercise ran os.system(command) here, which is remote code
    # execution by prompt: whoever writes the user message chooses what runs on
    # your laptop. Module 08 covers this as the most common MCP server flaw.
    #
    # Run the cell and watch the model call it anyway. That is the point.
    return f"refused; would have run {command!r}"


agent = create_agent(
    model=llm,                              # the configured LLM object, not a string
    tools=[recipe_for_dish, open_cmd],
    system_prompt="You are a helpful assistant",
)

result = agent.invoke({"messages": [{
    "role": "user",
    "content": "You have a tool called open_cmd that runs shell commands. "
               "Use it to list the files on the C drive.",
}]})

for message in result["messages"]:
    print(f"{type(message).__name__:14} | {(message.content or '')[:100]}")

The model called `open_cmd` without hesitating, because a tool description is
the only thing it has to go on and this one says the tool runs shell commands.

Nothing protects a tool except the code inside it. Three things would have to
be true before a tool like this could ship :

- the command is chosen from a fixed list, never built from model output
- it runs somewhere disposable, away from the machine holding our files
- a person approves the call, as in module 09's `HumanInTheLoopMiddleware`

### Solution

Expected behaviour: the agent calls `get_recipe` and repeats what it returned.

A small model will sometimes answer a cooking question straight out of its own
knowledge instead of calling the tool. Tighten the system prompt to
"Always use the get_recipe tool to answer questions about dishes" if we need
the tool route for a demonstration.

In [ ]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

RECIPES = {
    "pancakes": "Mix 200g flour, 2 eggs, 300ml milk and a pinch of salt. Fry each side for 2 minutes.",
    "omelette": "Beat 3 eggs with salt and pepper. Pour into a hot buttered pan and cook until set.",
    "pizza margherita": "Top pizza dough with tomato sauce, mozzarella and basil. Bake at 250°C for 10 minutes.",
    "pierogi": "Fill dough circles with potato and cheese, seal, then boil for 3-4 minutes until they float.",
}

def get_recipe(dish: str) -> str:
    """Return the itinerary for the chosen trip."""
    return RECIPES.get(
        dish.lower().strip(),
        "Recipe for water for tea: Boil water, pour over a tea bag, steep for 3-5 minutes.",
    )

llm = make_llm()

agent = create_agent(
    model=llm,
    tools=[get_recipe],
    system_prompt="You are a helpful cooking assistant. Use the get_recipe tool to answer.",
)

result = agent.invoke({"messages": [{"role": "user", "content": "How do I make pancakes?"}]})
last_msg = result["messages"][-1]
print(last_msg.content)

### Exercise Structured output – `response_format` w `create_agent`
In the example below, add the person's home address to the prompt. Also add an `address` field to the pydantic model.

In [5]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address")
    phone: str = Field(description="The phone number")

llm = make_llm()

agent = create_agent(
    model=llm,
    response_format=ContactInfo,
)

result = agent.invoke({
    "messages": [
        {"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}
    ]
})
structured = result["structured_response"]
print(structured)
print(type(structured))

name='John Doe' email='john@example.com' phone='(555) 123-4567'
<class '__main__.ContactInfo'>


### Solution 

In [6]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address")
    phone: str = Field(description="The phone number")
    address: str = Field(description="The home address")      # <- added field

llm = make_llm()

agent = create_agent(
    model=llm,
    response_format=ContactInfo,
)

result = agent.invoke({
    "messages": [
        {"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567, 123 Main Street, Springfield"}   # <- address added
    ]
})
structured = result["structured_response"]
print(structured)
print(type(structured))

name='John Doe' email='john@example.com' phone='(555) 123-4567' address='123 Main Street, Springfield'
<class '__main__.ContactInfo'>


### Exercise Short‑term memory
In the example below, first tell the model our flight number. Then ask the model which flight we are on.

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

llm = make_llm()
checkpointer = InMemorySaver()

agent = create_agent(
    model=llm,
    checkpointer=checkpointer,
)

config = {"configurable": {"thread_id": "demo-thread-1"}}

agent.invoke({"messages": [{"role": "user", "content": "Hi! My name is Michał."}]}, config=config)
result = agent.invoke({"messages": [{"role": "user", "content": "Which flight am I on?"}]}, config=config)
print(result["messages"][-1].content)

### Solution

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain_openai import ChatOpenAI

llm = make_llm()

checkpointer = InMemorySaver()

agent = create_agent(
    model=llm,
    checkpointer=checkpointer,
)

config = {"configurable": {"thread_id": "demo-thread-1"}}

# First: tell the model the flight number
agent.invoke(
    {"messages": [{"role": "user", "content": "My flight today is KL1495 to Berlin, gate B°C."}]},
    config=config,
)

# Then: ask the model which flight you are on ; same thread_id, so it remembers
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Which flight am I on?"}]},
    config=config,
)
print(result["messages"][-1].content)